# 1. MRI NIfTI preprocessing and quality control

This notebook starts from the converted NIfTI MRI files created in the previous notebook.

The cohort has already been fixed at 1063 selected baseline or near-baseline T1 MRI scans from ADNI. The DICOM extraction and DICOM-to-NIfTI conversion steps are complete, and all converted NIfTI files passed the initial file-level and header-level quality checks.

The purpose of this notebook is to begin MRI preprocessing and standardization. The raw converted NIfTI files may still differ in orientation, image shape, voxel size, scanner protocol, and intensity scale. These differences need to be checked and standardized before the images can be used for machine learning.

## 1.1. Preprocessing Plan

The first preprocessing checks will focus on standardizing the spatial structure of the MRI files.

The planned steps are:

1. Load the manifest and saved NIfTI QC reports.
2. Inspect the small number of non-RAS images.
3. Reorient all NIfTI files to a common RAS orientation.
4. Save reoriented files into a separate preprocessing folder.
5. Create QC reports confirming that all files were reoriented successfully.
6. Continue with later preprocessing decisions such as resampling, registration, cropping or resizing, and intensity normalization.

This notebook does not rebuild the cohort, reselect MRI scans, or redo DICOM-to-NIfTI conversion.

In [ ]:
import json

In [ ]:
from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np

try:
    import nibabel as nib
except ImportError:
    !pip -q install nibabel
    import nibabel as nib

drive.mount("/content/drive", force_remount=True)

BASE_DIR = Path("/content/drive/My Drive/adni_mri")

MANIFEST_DIR = BASE_DIR / "manifest"
QC_DIR = BASE_DIR / "qc"

NIFTI_DIR = BASE_DIR / "nifti"
PROCESSED_DIR = BASE_DIR / "processed"
REORIENTED_NIFTI_DIR = PROCESSED_DIR / "nifti_ras"

EXTRACTION_MANIFEST_PATH = MANIFEST_DIR / "clean_90d_extraction_manifest_1063.csv"
NIFTI_FILE_QC_REPORT_PATH = QC_DIR / "clean_90d_nifti_file_qc_report.csv"
NIFTI_HEADER_QC_REPORT_PATH = QC_DIR / "clean_90d_nifti_header_qc_report.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REORIENTED_NIFTI_DIR.mkdir(parents=True, exist_ok=True)

print("Base folder exists:", BASE_DIR.exists())
print("Manifest folder exists:", MANIFEST_DIR.exists())
print("QC folder exists:", QC_DIR.exists())
print("NIfTI folder exists:", NIFTI_DIR.exists())
print("Processed folder exists:", PROCESSED_DIR.exists())
print("Reoriented NIfTI folder exists:", REORIENTED_NIFTI_DIR.exists())

print("\nRequired input files:")
print("Extraction manifest:", EXTRACTION_MANIFEST_PATH.exists())
print("NIfTI file QC report:", NIFTI_FILE_QC_REPORT_PATH.exists())
print("NIfTI header QC report:", NIFTI_HEADER_QC_REPORT_PATH.exists())

print("\nNIfTI group folders:")
for folder in sorted(NIFTI_DIR.iterdir()):
    if folder.is_dir():
        print("-", folder)

## 1.2. Load Manifest and NIfTI QC Reports

This notebook starts from the saved manifest and QC reports created in the previous notebooks. These files define the clean 90-day MRI cohort and provide the file paths and header metadata for the converted NIfTI images.

The reports are loaded here so that preprocessing can continue from the validated NIfTI dataset without repeating DICOM extraction or DICOM-to-NIfTI conversion.

In [ ]:
extraction_manifest = pd.read_csv(EXTRACTION_MANIFEST_PATH)
nifti_file_qc = pd.read_csv(NIFTI_FILE_QC_REPORT_PATH)
nifti_header_qc = pd.read_csv(NIFTI_HEADER_QC_REPORT_PATH)

print("Extraction manifest shape:", extraction_manifest.shape)
print("NIfTI file QC shape:", nifti_file_qc.shape)
print("NIfTI header QC shape:", nifti_header_qc.shape)

print("\nFinal group counts:")
print(extraction_manifest["final_group"].value_counts())

print("\nHeader QC status counts:")
print(nifti_header_qc["header_qc_status"].value_counts())

print("\nOrientation counts:")
print(nifti_header_qc["orientation"].value_counts())

print("\nPreview:")
display(nifti_header_qc.head())

## 1.3. Inspect Non-RAS NIfTI Images

The NIfTI header QC showed that most images are already stored in RAS orientation, but seven images have a different orientation. These images are still valid 3D MRI volumes, but they should be documented before spatial standardization.

This step identifies the non-RAS images and saves a small QC report. The images are not removed at this stage. They will later be reoriented into a common orientation together with the rest of the cohort.

In [ ]:
NON_RAS_NIFTI_REPORT_PATH = QC_DIR / "clean_90d_non_ras_nifti_images.csv"

# Reload header QC if needed
if "nifti_header_qc" not in globals():
    nifti_header_qc = pd.read_csv(NIFTI_HEADER_QC_REPORT_PATH)

non_ras_images = nifti_header_qc[
    nifti_header_qc["orientation"] != "RAS"
].copy()

# Add useful manifest information for interpretation
manifest_context_cols = [
    "RID",
    "subject_id",
    "final_group",
    "image_id",
    "baseline_phase",
    "study_date",
    "days_from_baseline_mri",
    "description",
    "visit",
    "phase",
    "research_group",
    "field_strength",
    "manufacturer_clean",
    "model_family",
    "source_zip",
    "total_dcm_file_count"
]

non_ras_images = non_ras_images.merge(
    extraction_manifest[manifest_context_cols],
    on=["RID", "subject_id", "final_group", "image_id"],
    how="left"
)

non_ras_images.to_csv(NON_RAS_NIFTI_REPORT_PATH, index=False)

print("Saved non-RAS NIfTI report:")
print(NON_RAS_NIFTI_REPORT_PATH)

print("\nNumber of non-RAS images:")
print(len(non_ras_images))

print("\nOrientation counts among non-RAS images:")
print(non_ras_images["orientation"].value_counts())

print("\nNon-RAS images:")
display(
    non_ras_images[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "orientation",
            "shape",
            "voxel_sizes",
            "description",
            "visit",
            "phase",
            "field_strength",
            "manufacturer_clean",
            "model_family",
            "nifti_path"
        ]
    ].sort_values(["orientation", "final_group", "RID"])
)

Note: I noticed that field strength is missing

## 1.4. Check Scanner Metadata from NIfTI JSON Files

Some images have missing `field_strength` values in the extraction manifest. This does not necessarily indicate a problem with the MRI scan. The field strength values in the manifest came from earlier IDA metadata tables, and some rows did not have scanner metadata available during manifest construction.

After DICOM-to-NIfTI conversion, `dcm2niix` created JSON sidecar files for each NIfTI image. These JSON files may contain scanner metadata extracted directly from the DICOM headers, including magnetic field strength, manufacturer, scanner model, protocol name, and series description.

This step checks whether field strength and scanner metadata can be recovered from the NIfTI JSON sidecar files.

In [ ]:
JSON_SCANNER_METADATA_REPORT_PATH = QC_DIR / "clean_90d_json_scanner_metadata_report.csv"

# Reload file-level QC if needed
if "nifti_file_qc" not in globals():
    nifti_file_qc = pd.read_csv(NIFTI_FILE_QC_REPORT_PATH)

json_metadata_records = []

for _, row in nifti_file_qc.iterrows():
    json_path = row["primary_json_path"]

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "json_path": json_path,
        "json_loaded": False,
        "MagneticFieldStrength": np.nan,
        "Manufacturer": np.nan,
        "ManufacturersModelName": np.nan,
        "ProtocolName": np.nan,
        "SeriesDescription": np.nan,
        "ScanningSequence": np.nan,
        "SequenceVariant": np.nan,
        "MRAcquisitionType": np.nan,
        "error_message": ""
    }

    try:
        with open(json_path, "r") as f:
            metadata = json.load(f)

        record["json_loaded"] = True
        record["MagneticFieldStrength"] = metadata.get("MagneticFieldStrength", np.nan)
        record["Manufacturer"] = metadata.get("Manufacturer", np.nan)
        record["ManufacturersModelName"] = metadata.get("ManufacturersModelName", np.nan)
        record["ProtocolName"] = metadata.get("ProtocolName", np.nan)
        record["SeriesDescription"] = metadata.get("SeriesDescription", np.nan)
        record["ScanningSequence"] = metadata.get("ScanningSequence", np.nan)
        record["SequenceVariant"] = metadata.get("SequenceVariant", np.nan)
        record["MRAcquisitionType"] = metadata.get("MRAcquisitionType", np.nan)

    except Exception as e:
        record["error_message"] = str(e)

    json_metadata_records.append(record)

json_scanner_metadata = pd.DataFrame(json_metadata_records)

json_scanner_metadata.to_csv(JSON_SCANNER_METADATA_REPORT_PATH, index=False)

print("Saved JSON scanner metadata report:")
print(JSON_SCANNER_METADATA_REPORT_PATH)

print("\nJSON loaded counts:")
print(json_scanner_metadata["json_loaded"].value_counts(dropna=False))

print("\nMagnetic field strength counts from JSON:")
print(json_scanner_metadata["MagneticFieldStrength"].value_counts(dropna=False))

print("\nManufacturer counts from JSON:")
print(json_scanner_metadata["Manufacturer"].value_counts(dropna=False))

print("\nScanner model counts from JSON:")
print(json_scanner_metadata["ManufacturersModelName"].value_counts(dropna=False).head(20))

print("\nMissing field strength in original extraction manifest:")
print(extraction_manifest["field_strength"].isna().sum())

print("\nMissing field strength in JSON metadata:")
print(json_scanner_metadata["MagneticFieldStrength"].isna().sum())

display(json_scanner_metadata.head())

## 1.5. Standardize Scanner Metadata from JSON Sidecar Files

The JSON sidecar files created during DICOM-to-NIfTI conversion contain scanner metadata extracted from the original DICOM headers. This recovered the missing field strength values from the original extraction manifest.

The raw `MagneticFieldStrength` values are not perfectly standardized. Some values are stored as Tesla-like values, such as `1.5`, `1.494`, `2.89362`, and `3.0`, while a small number are stored as larger values such as `15000` and `30000`. These larger values correspond to the same scanner strengths but appear to be stored in a different unit scale.

For analysis and QC, these raw values are standardized into a clean scanner field strength category:

- values close to 1.5 are treated as `1.5T`
- values close to 3.0 are treated as `3T`
- values such as 15000 and 30000 are converted to 1.5T and 3T respectively

This creates a cleaner scanner metadata table that can be used for later cohort description, scanner-bias checks, and modelling QC.

In [ ]:
CLEAN_SCANNER_METADATA_REPORT_PATH = QC_DIR / "clean_90d_standardized_scanner_metadata.csv"
CLEAN_MANIFEST_WITH_SCANNER_METADATA_PATH = MANIFEST_DIR / "clean_90d_extraction_manifest_1063_with_json_scanner_metadata.csv"

def standardize_field_strength_t(value):
    """
    Standardize raw magnetic field strength values into Tesla.

    Most values are already close to Tesla values, such as:
        1.494, 1.5, 2.89362, 3.0

    A few values appear in a larger unit scale:
        15000 -> 1.5T
        30000 -> 3T
    """
    if pd.isna(value):
        return np.nan

    value = float(value)

    # Convert very large values to Tesla-like units.
    # 15000 becomes 1.5 and 30000 becomes 3.0.
    if value > 1000:
        value = value / 10000

    # Map near-1.5T values to exactly 1.5.
    if abs(value - 1.5) <= 0.15:
        return 1.5

    # Map near-3T values to exactly 3.0.
    if abs(value - 3.0) <= 0.20:
        return 3.0

    return value

scanner_metadata_clean = json_scanner_metadata.copy()

scanner_metadata_clean["field_strength_json_raw"] = scanner_metadata_clean["MagneticFieldStrength"]
scanner_metadata_clean["field_strength_t_clean"] = scanner_metadata_clean[
    "MagneticFieldStrength"
].apply(standardize_field_strength_t)

scanner_metadata_clean["field_strength_category"] = scanner_metadata_clean[
    "field_strength_t_clean"
].map({
    1.5: "1.5T",
    3.0: "3T"
})

scanner_metadata_clean["manufacturer_json"] = scanner_metadata_clean["Manufacturer"]
scanner_metadata_clean["scanner_model_json"] = scanner_metadata_clean["ManufacturersModelName"]
scanner_metadata_clean["protocol_name_json"] = scanner_metadata_clean["ProtocolName"]
scanner_metadata_clean["series_description_json"] = scanner_metadata_clean["SeriesDescription"]

scanner_metadata_clean.to_csv(CLEAN_SCANNER_METADATA_REPORT_PATH, index=False)

scanner_metadata_for_merge = scanner_metadata_clean[
    [
        "RID",
        "subject_id",
        "final_group",
        "image_id",
        "field_strength_json_raw",
        "field_strength_t_clean",
        "field_strength_category",
        "manufacturer_json",
        "scanner_model_json",
        "protocol_name_json",
        "series_description_json"
    ]
].copy()

manifest_with_json_scanner_metadata = extraction_manifest.merge(
    scanner_metadata_for_merge,
    on=["RID", "subject_id", "final_group", "image_id"],
    how="left"
)

manifest_with_json_scanner_metadata.to_csv(
    CLEAN_MANIFEST_WITH_SCANNER_METADATA_PATH,
    index=False
)

print("Saved standardized scanner metadata report:")
print(CLEAN_SCANNER_METADATA_REPORT_PATH)

print("\nSaved manifest with JSON scanner metadata:")
print(CLEAN_MANIFEST_WITH_SCANNER_METADATA_PATH)

print("\nRaw field strength counts:")
print(scanner_metadata_clean["field_strength_json_raw"].value_counts(dropna=False))

print("\nClean field strength counts:")
print(scanner_metadata_clean["field_strength_category"].value_counts(dropna=False))

print("\nClean field strength by final group:")
display(
    manifest_with_json_scanner_metadata
    .groupby(["final_group", "field_strength_category"])
    .size()
    .unstack(fill_value=0)
)

print("\nManufacturer counts from JSON:")
print(scanner_metadata_clean["manufacturer_json"].value_counts(dropna=False))

print("\nMissing clean field strength values:")
print(manifest_with_json_scanner_metadata["field_strength_category"].isna().sum())

display(
    manifest_with_json_scanner_metadata[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "field_strength",
            "field_strength_json_raw",
            "field_strength_category",
            "manufacturer_clean",
            "manufacturer_json",
            "model_family",
            "scanner_model_json"
        ]
    ].head()
)

## 1.6. Interpretation of Non-RAS Image Check

The non-RAS image check identified 7 NIfTI files whose orientation is not RAS.

Most converted images are already stored in RAS orientation, but 6 images are stored as PSR and 1 image is stored as PIR. These files are not corrupted or invalid, because they were successfully loaded during the NIfTI header QC and were confirmed to be valid 3D MRI volumes.

This check is important because machine learning models should not receive images with inconsistent spatial orientation. If two images store the anatomical axes differently, the same brain structure may appear in different array directions. Therefore, all images should be reoriented to a common orientation before later preprocessing and modelling.

The 7 non-RAS files will not be removed. Instead, all 1063 NIfTI files will be saved into a new standardized folder after being reoriented to RAS. The original converted NIfTI files will remain unchanged.

## 1.7. Reorient All NIfTI Files to RAS

This step reorients all converted NIfTI images to a common RAS orientation.

The original converted NIfTI files are not overwritten. Instead, the reoriented files are saved into a separate preprocessing folder. This keeps the workflow traceable and allows the original conversion outputs to be preserved.

Images that are already in RAS orientation are still copied into the standardized folder, so the output folder contains one RAS-standardized NIfTI file for every selected MRI scan.

In [ ]:
import time

RAS_REORIENTATION_REPORT_PATH = QC_DIR / "clean_90d_ras_reorientation_report.csv"

# Reload header QC if needed
if "nifti_header_qc" not in globals():
    nifti_header_qc = pd.read_csv(NIFTI_HEADER_QC_REPORT_PATH)

REORIENTED_NIFTI_DIR.mkdir(parents=True, exist_ok=True)

for group in sorted(nifti_header_qc["final_group"].unique()):
    (REORIENTED_NIFTI_DIR / group).mkdir(parents=True, exist_ok=True)

reorientation_records = []

start_time = time.time()

for idx, row in nifti_header_qc.iterrows():
    subject_id = row["subject_id"]
    final_group = row["final_group"]
    image_id = int(row["image_id"])
    input_path = Path(row["nifti_path"])

    output_dir = REORIENTED_NIFTI_DIR / final_group
    output_path = output_dir / input_path.name

    record = {
        "RID": row["RID"],
        "subject_id": subject_id,
        "final_group": final_group,
        "image_id": image_id,
        "input_nifti_path": str(input_path),
        "output_nifti_path": str(output_path),
        "input_orientation": row["orientation"],
        "output_orientation": None,
        "input_shape": row["shape"],
        "output_shape": None,
        "reorientation_status": None,
        "error_message": ""
    }

    # Make the step resumable.
    # If the output file already exists, check it instead of recreating it.
    if output_path.exists():
        try:
            existing_img = nib.load(str(output_path))
            record["output_orientation"] = "".join(nib.aff2axcodes(existing_img.affine))
            record["output_shape"] = str(existing_img.shape)
            record["reorientation_status"] = "skipped_existing_output"
        except Exception as e:
            record["reorientation_status"] = "existing_output_load_failed"
            record["error_message"] = str(e)

        reorientation_records.append(record)
        continue

    try:
        img = nib.load(str(input_path))

        # Convert the image to the closest canonical orientation.
        # In nibabel, this usually means RAS orientation.
        ras_img = nib.as_closest_canonical(img)

        nib.save(ras_img, str(output_path))

        saved_img = nib.load(str(output_path))

        record["output_orientation"] = "".join(nib.aff2axcodes(saved_img.affine))
        record["output_shape"] = str(saved_img.shape)
        record["reorientation_status"] = "reoriented_saved"

    except Exception as e:
        record["reorientation_status"] = "reorientation_failed"
        record["error_message"] = str(e)

    reorientation_records.append(record)

    if len(reorientation_records) % 100 == 0:
        pd.DataFrame(reorientation_records).to_csv(
            RAS_REORIENTATION_REPORT_PATH,
            index=False
        )
        elapsed_minutes = (time.time() - start_time) / 60
        print(f"Progress saved after {len(reorientation_records)} images. Elapsed minutes: {elapsed_minutes:.2f}")

ras_reorientation_report = pd.DataFrame(reorientation_records)
ras_reorientation_report.to_csv(RAS_REORIENTATION_REPORT_PATH, index=False)

elapsed_minutes = (time.time() - start_time) / 60

print("Saved RAS reorientation report:")
print(RAS_REORIENTATION_REPORT_PATH)

print("\nReorientation status counts:")
print(ras_reorientation_report["reorientation_status"].value_counts(dropna=False))

print("\nOutput orientation counts:")
print(ras_reorientation_report["output_orientation"].value_counts(dropna=False))

print("\nInput orientation counts:")
print(ras_reorientation_report["input_orientation"].value_counts(dropna=False))

print("\nOutput files created/existing:")
print(ras_reorientation_report["output_nifti_path"].apply(lambda x: Path(x).exists()).value_counts())

print("\nElapsed time in minutes:")
print(round(elapsed_minutes, 2))

problem_reorientation = ras_reorientation_report[
    (ras_reorientation_report["output_orientation"] != "RAS") |
    (~ras_reorientation_report["output_nifti_path"].apply(lambda x: Path(x).exists()))
].copy()

print("\nProblem reorientation outputs:")
print(len(problem_reorientation))

if len(problem_reorientation) > 0:
    display(
        problem_reorientation[
            [
                "RID",
                "subject_id",
                "final_group",
                "image_id",
                "input_orientation",
                "output_orientation",
                "reorientation_status",
                "error_message",
                "input_nifti_path",
                "output_nifti_path"
            ]
        ]
    )

print("\nAll output images are RAS:")
print((ras_reorientation_report["output_orientation"] == "RAS").all())

## 1.8. Validate RAS Reorientation at the Voxel-Array Level

The reorientation step saved a RAS-standardized NIfTI file for every selected MRI scan. A basic orientation check can confirm that the output files report RAS orientation, but this does not fully prove that the non-RAS images were actually reordered or flipped at the voxel-array level.

This validation focuses on the images that were originally not RAS. For each of these images, the expected RAS-oriented image is recomputed from the original NIfTI file using `nib.as_closest_canonical`. The recomputed voxel array and affine matrix are then compared with the saved RAS output file.

This confirms whether the saved output matches the expected RAS-reoriented data, rather than only having updated orientation labels in the header.

In [ ]:
RAS_ARRAY_VALIDATION_REPORT_PATH = QC_DIR / "clean_90d_ras_array_validation_report.csv"

# Reload the reorientation report if needed.
# This is the report created by the previous RAS reorientation step.
if "ras_reorientation_report" not in globals():
    ras_reorientation_report = pd.read_csv(RAS_REORIENTATION_REPORT_PATH)

# Only images that were originally non-RAS need the stronger voxel-array validation.
non_ras_reorientation = ras_reorientation_report[
    ras_reorientation_report["input_orientation"] != "RAS"
].copy()

print("Originally non-RAS images to validate:", len(non_ras_reorientation))

array_validation_records = []

for _, row in non_ras_reorientation.iterrows():
    input_path = Path(row["input_nifti_path"])
    output_path = Path(row["output_nifti_path"])

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "input_orientation": row["input_orientation"],
        "reported_output_orientation": row["output_orientation"],
        "input_nifti_path": str(input_path),
        "output_nifti_path": str(output_path),
        "input_shape": None,
        "expected_ras_shape": None,
        "saved_output_shape": None,
        "expected_ras_orientation": None,
        "saved_output_orientation": None,
        "voxel_array_matches_expected_ras": False,
        "affine_matches_expected_ras": False,
        "validation_status": None,
        "error_message": ""
    }

    try:
        # Load the original input and the saved RAS output.
        input_img = nib.load(str(input_path))
        saved_output_img = nib.load(str(output_path))

        # Recompute the expected RAS-oriented image from the original input.
        # This is the reference result that the saved output should match.
        expected_ras_img = nib.as_closest_canonical(input_img)

        expected_ras_data = np.asanyarray(expected_ras_img.dataobj)
        saved_output_data = np.asanyarray(saved_output_img.dataobj)

        record["input_shape"] = str(input_img.shape)
        record["expected_ras_shape"] = str(expected_ras_img.shape)
        record["saved_output_shape"] = str(saved_output_img.shape)

        record["expected_ras_orientation"] = "".join(
            nib.aff2axcodes(expected_ras_img.affine)
        )
        record["saved_output_orientation"] = "".join(
            nib.aff2axcodes(saved_output_img.affine)
        )

        record["voxel_array_matches_expected_ras"] = np.array_equal(
            expected_ras_data,
            saved_output_data
        )

        record["affine_matches_expected_ras"] = np.allclose(
            expected_ras_img.affine,
            saved_output_img.affine
        )

        if (
            record["saved_output_orientation"] == "RAS"
            and record["voxel_array_matches_expected_ras"]
            and record["affine_matches_expected_ras"]
        ):
            record["validation_status"] = "array_and_affine_match_expected_ras"
        else:
            record["validation_status"] = "check_reorientation_output"

    except Exception as e:
        record["validation_status"] = "validation_failed"
        record["error_message"] = str(e)

    array_validation_records.append(record)

ras_array_validation_report = pd.DataFrame(array_validation_records)
ras_array_validation_report.to_csv(RAS_ARRAY_VALIDATION_REPORT_PATH, index=False)

print("Saved RAS array validation report:")
print(RAS_ARRAY_VALIDATION_REPORT_PATH)

print("\nValidation status counts:")
print(ras_array_validation_report["validation_status"].value_counts(dropna=False))

print("\nVoxel array matches expected RAS:")
print(ras_array_validation_report["voxel_array_matches_expected_ras"].value_counts(dropna=False))

print("\nAffine matches expected RAS:")
print(ras_array_validation_report["affine_matches_expected_ras"].value_counts(dropna=False))

print("\nValidation details:")
display(
    ras_array_validation_report[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "input_orientation",
            "expected_ras_orientation",
            "saved_output_orientation",
            "input_shape",
            "expected_ras_shape",
            "saved_output_shape",
            "voxel_array_matches_expected_ras",
            "affine_matches_expected_ras",
            "validation_status"
        ]
    ]
)

print("\nAll originally non-RAS images passed voxel-array validation:")
print(
    (
        ras_array_validation_report["validation_status"]
        == "array_and_affine_match_expected_ras"
    ).all()
)

## 1.9. Diagnose Voxel-Array Validation Differences

The initial array-level validation used exact equality, which is stricter than necessary for saved and reloaded NIfTI files. NIfTI writing may preserve the image correctly while still causing small numerical differences due to datatype, scaling, or compression-related save/load behavior.

This step performs a more informative comparison for the originally non-RAS images. It compares the recomputed expected RAS image with the saved RAS output using numerical closeness, and records the maximum and mean absolute voxel differences.

In [ ]:
RAS_ARRAY_DIAGNOSTIC_REPORT_PATH = QC_DIR / "clean_90d_ras_array_diagnostic_report.csv"

if "ras_array_validation_report" not in globals():
    ras_array_validation_report = pd.read_csv(RAS_ARRAY_VALIDATION_REPORT_PATH)

array_diagnostic_records = []

for _, row in ras_array_validation_report.iterrows():
    input_path = Path(row["input_nifti_path"])
    output_path = Path(row["output_nifti_path"])

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "input_orientation": row["input_orientation"],
        "input_nifti_path": str(input_path),
        "output_nifti_path": str(output_path),
        "expected_ras_orientation": None,
        "saved_output_orientation": None,
        "expected_ras_shape": None,
        "saved_output_shape": None,
        "expected_dtype": None,
        "saved_dtype": None,
        "exact_array_equal": None,
        "array_allclose": None,
        "max_abs_difference": None,
        "mean_abs_difference": None,
        "affine_allclose": None,
        "diagnostic_status": None,
        "error_message": ""
    }

    try:
        input_img = nib.load(str(input_path))
        saved_img = nib.load(str(output_path))
        expected_ras_img = nib.as_closest_canonical(input_img)

        expected_data = expected_ras_img.get_fdata(dtype=np.float32)
        saved_data = saved_img.get_fdata(dtype=np.float32)

        abs_diff = np.abs(expected_data - saved_data)

        record["expected_ras_orientation"] = "".join(nib.aff2axcodes(expected_ras_img.affine))
        record["saved_output_orientation"] = "".join(nib.aff2axcodes(saved_img.affine))
        record["expected_ras_shape"] = str(expected_ras_img.shape)
        record["saved_output_shape"] = str(saved_img.shape)
        record["expected_dtype"] = str(expected_ras_img.get_data_dtype())
        record["saved_dtype"] = str(saved_img.get_data_dtype())

        record["exact_array_equal"] = np.array_equal(expected_data, saved_data)
        record["array_allclose"] = np.allclose(expected_data, saved_data, rtol=1e-5, atol=1e-3)
        record["max_abs_difference"] = float(np.max(abs_diff))
        record["mean_abs_difference"] = float(np.mean(abs_diff))
        record["affine_allclose"] = np.allclose(expected_ras_img.affine, saved_img.affine)

        if (
            record["saved_output_orientation"] == "RAS"
            and record["expected_ras_shape"] == record["saved_output_shape"]
            and record["array_allclose"]
            and record["affine_allclose"]
        ):
            record["diagnostic_status"] = "passed_numeric_validation"
        else:
            record["diagnostic_status"] = "needs_rewrite_or_visual_check"

    except Exception as e:
        record["diagnostic_status"] = "diagnostic_failed"
        record["error_message"] = str(e)

    array_diagnostic_records.append(record)

ras_array_diagnostic_report = pd.DataFrame(array_diagnostic_records)
ras_array_diagnostic_report.to_csv(RAS_ARRAY_DIAGNOSTIC_REPORT_PATH, index=False)

print("Saved RAS array diagnostic report:")
print(RAS_ARRAY_DIAGNOSTIC_REPORT_PATH)

print("\nDiagnostic status counts:")
print(ras_array_diagnostic_report["diagnostic_status"].value_counts(dropna=False))

print("\nExact array equality:")
print(ras_array_diagnostic_report["exact_array_equal"].value_counts(dropna=False))

print("\nNumerical allclose:")
print(ras_array_diagnostic_report["array_allclose"].value_counts(dropna=False))

print("\nAffine allclose:")
print(ras_array_diagnostic_report["affine_allclose"].value_counts(dropna=False))

print("\nMaximum absolute difference summary:")
display(ras_array_diagnostic_report["max_abs_difference"].describe())

print("\nDetailed diagnostic table:")
display(
    ras_array_diagnostic_report[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "input_orientation",
            "expected_ras_orientation",
            "saved_output_orientation",
            "expected_ras_shape",
            "saved_output_shape",
            "expected_dtype",
            "saved_dtype",
            "exact_array_equal",
            "array_allclose",
            "max_abs_difference",
            "mean_abs_difference",
            "affine_allclose",
            "diagnostic_status"
        ]
    ]
)

print("\nAll originally non-RAS images passed numeric voxel-array validation:")
print(
    (
        ras_array_diagnostic_report["diagnostic_status"]
        == "passed_numeric_validation"
    ).all()
)

## 1.10. Visual Check of RAS Reorientation Outputs

The numerical validation showed that 2 originally non-RAS images matched the expected RAS output exactly or numerically, while 5 did not. Before rewriting the failed outputs, I visually inspect the images.

For each originally non-RAS image, this step compares the expected RAS image recomputed from the original NIfTI file with the saved RAS output image. The middle axial slice is displayed for both versions side by side. This helps determine whether the failed numerical checks correspond to visible anatomical differences or only small intensity/scaling differences.

In [ ]:
import matplotlib.pyplot as plt

# Use the diagnostic report that contains the 2 passed and 5 failed cases.
if "ras_array_diagnostic_report" not in globals():
    ras_array_diagnostic_report = pd.read_csv(RAS_ARRAY_DIAGNOSTIC_REPORT_PATH)

visual_check_cases = ras_array_diagnostic_report.copy()

print("Cases to visually inspect:", len(visual_check_cases))
print("\nDiagnostic status counts:")
print(visual_check_cases["diagnostic_status"].value_counts())

def get_middle_axial_slice(data):
    """
    Return the middle slice along the third axis.
    This is a simple axial-style view for quick visual QC.
    """
    z_index = data.shape[2] // 2
    return data[:, :, z_index]

def normalize_for_display(slice_2d):
    """
    Normalize a 2D image slice for display using robust percentiles.
    This only affects visualization, not the saved MRI data.
    """
    lower, upper = np.percentile(slice_2d, [1, 99])

    if upper <= lower:
        return slice_2d

    normalized = (slice_2d - lower) / (upper - lower)
    normalized = np.clip(normalized, 0, 1)

    return normalized

for _, row in visual_check_cases.iterrows():
    input_path = Path(row["input_nifti_path"])
    output_path = Path(row["output_nifti_path"])

    input_img = nib.load(str(input_path))
    expected_ras_img = nib.as_closest_canonical(input_img)
    saved_output_img = nib.load(str(output_path))

    expected_data = expected_ras_img.get_fdata(dtype=np.float32)
    saved_data = saved_output_img.get_fdata(dtype=np.float32)

    expected_slice = normalize_for_display(get_middle_axial_slice(expected_data))
    saved_slice = normalize_for_display(get_middle_axial_slice(saved_data))
    diff_slice = np.abs(expected_slice - saved_slice)

    max_abs_difference = row["max_abs_difference"]
    mean_abs_difference = row["mean_abs_difference"]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(np.rot90(expected_slice), cmap="gray")
    axes[0].set_title("Expected RAS")
    axes[0].axis("off")

    axes[1].imshow(np.rot90(saved_slice), cmap="gray")
    axes[1].set_title("Saved RAS Output")
    axes[1].axis("off")

    axes[2].imshow(np.rot90(diff_slice), cmap="gray")
    axes[2].set_title("Absolute Difference")
    axes[2].axis("off")

    fig.suptitle(
        f"{row['subject_id']} | I{int(row['image_id'])} | "
        f"{row['input_orientation']} → RAS | "
        f"{row['diagnostic_status']} | "
        f"max diff={max_abs_difference:.3f}, mean diff={mean_abs_difference:.3f}",
        fontsize=10
    )

    plt.tight_layout()
    plt.show()

## 1.11. Interpretation of Visual RAS Reorientation Check

The visual RAS reorientation check compared the expected RAS image with the saved RAS output for the originally non-RAS NIfTI files.

The expected RAS image was recomputed directly from the original converted NIfTI file using `nib.as_closest_canonical`. The saved RAS output was the file previously written into the standardized `processed/nifti_ras` folder. Therefore, the comparison checked whether the saved output visually matched the RAS version that should have been produced from the original image.

The visual inspection showed that the expected RAS images and saved RAS outputs are anatomically aligned. The brain structures appear in the same orientation and position, with no obvious left-right flip, rotation error, or slice-order mismatch. This suggests that the RAS reorientation step corrected the spatial orientation of the non-RAS images.

However, the numerical validation showed that 5 of the 7 originally non-RAS images did not match the expected RAS voxel array exactly or approximately. The absolute-difference images showed intensity differences across the brain rather than clear spatial misalignment. Since the affine matrices matched for all 7 images and the visual anatomy appeared aligned, the issue is most likely related to intensity scaling, datatype handling, or save/reload behavior rather than failed spatial reorientation.

To keep the preprocessing outputs clean and reproducible, the 5 failed RAS outputs should be rewritten directly from the expected canonical RAS images. This does not remove any subjects and does not change the clinical labels. It only replaces the affected standardized RAS output files with explicitly saved canonical RAS versions.


In [ ]:
RAS_REWRITE_FAILED_REPORT_PATH = QC_DIR / "clean_90d_ras_rewrite_failed_non_ras_report.csv"

# Reload diagnostic report if needed
if "ras_array_diagnostic_report" not in globals():
    ras_array_diagnostic_report = pd.read_csv(RAS_ARRAY_DIAGNOSTIC_REPORT_PATH)

failed_non_ras = ras_array_diagnostic_report[
    ras_array_diagnostic_report["diagnostic_status"] == "needs_rewrite_or_visual_check"
].copy()

print("Failed non-RAS outputs to rewrite:", len(failed_non_ras))

rewrite_records = []

for _, row in failed_non_ras.iterrows():
    input_path = Path(row["input_nifti_path"])
    output_path = Path(row["output_nifti_path"])

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "input_orientation": row["input_orientation"],
        "input_nifti_path": str(input_path),
        "output_nifti_path": str(output_path),
        "rewrite_status": None,
        "saved_output_orientation": None,
        "expected_ras_shape": None,
        "saved_output_shape": None,
        "array_allclose_after_rewrite": False,
        "max_abs_difference_after_rewrite": None,
        "mean_abs_difference_after_rewrite": None,
        "affine_allclose_after_rewrite": False,
        "error_message": ""
    }

    try:
        # Load the original non-RAS image.
        input_img = nib.load(str(input_path))

        # Recompute the expected canonical RAS image.
        expected_ras_img = nib.as_closest_canonical(input_img)

        # Materialize the RAS voxel data as float32.
        # This avoids hidden scaling differences when saving and reloading.
        expected_ras_data = expected_ras_img.get_fdata(dtype=np.float32)

        # Save a clean RAS output using the expected RAS data and affine.
        rewritten_img = nib.Nifti1Image(
            expected_ras_data,
            expected_ras_img.affine
        )
        rewritten_img.header.set_data_dtype(np.float32)
        rewritten_img.set_qform(expected_ras_img.affine, code=1)
        rewritten_img.set_sform(expected_ras_img.affine, code=1)

        # Overwrite only the failed RAS-standardized output file.
        nib.save(rewritten_img, str(output_path))

        # Reload the rewritten output and validate it.
        saved_img = nib.load(str(output_path))
        saved_data = saved_img.get_fdata(dtype=np.float32)

        abs_diff = np.abs(expected_ras_data - saved_data)

        record["saved_output_orientation"] = "".join(nib.aff2axcodes(saved_img.affine))
        record["expected_ras_shape"] = str(expected_ras_img.shape)
        record["saved_output_shape"] = str(saved_img.shape)
        record["array_allclose_after_rewrite"] = np.allclose(
            expected_ras_data,
            saved_data,
            rtol=1e-5,
            atol=1e-3
        )
        record["max_abs_difference_after_rewrite"] = float(np.max(abs_diff))
        record["mean_abs_difference_after_rewrite"] = float(np.mean(abs_diff))
        record["affine_allclose_after_rewrite"] = np.allclose(
            expected_ras_img.affine,
            saved_img.affine
        )

        if (
            record["saved_output_orientation"] == "RAS"
            and record["expected_ras_shape"] == record["saved_output_shape"]
            and record["array_allclose_after_rewrite"]
            and record["affine_allclose_after_rewrite"]
        ):
            record["rewrite_status"] = "rewritten_and_validated"
        else:
            record["rewrite_status"] = "rewrite_still_needs_check"

    except Exception as e:
        record["rewrite_status"] = "rewrite_failed"
        record["error_message"] = str(e)

    rewrite_records.append(record)

ras_rewrite_failed_report = pd.DataFrame(rewrite_records)
ras_rewrite_failed_report.to_csv(RAS_REWRITE_FAILED_REPORT_PATH, index=False)

print("Saved rewrite report:")
print(RAS_REWRITE_FAILED_REPORT_PATH)

print("\nRewrite status counts:")
print(ras_rewrite_failed_report["rewrite_status"].value_counts(dropna=False))

print("\nOutput orientation after rewrite:")
print(ras_rewrite_failed_report["saved_output_orientation"].value_counts(dropna=False))

print("\nArray allclose after rewrite:")
print(ras_rewrite_failed_report["array_allclose_after_rewrite"].value_counts(dropna=False))

print("\nAffine allclose after rewrite:")
print(ras_rewrite_failed_report["affine_allclose_after_rewrite"].value_counts(dropna=False))

print("\nMaximum absolute difference after rewrite:")
display(ras_rewrite_failed_report["max_abs_difference_after_rewrite"].describe())

display(
    ras_rewrite_failed_report[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "input_orientation",
            "saved_output_orientation",
            "expected_ras_shape",
            "saved_output_shape",
            "array_allclose_after_rewrite",
            "max_abs_difference_after_rewrite",
            "mean_abs_difference_after_rewrite",
            "affine_allclose_after_rewrite",
            "rewrite_status"
        ]
    ]
)

print("\nAll failed non-RAS outputs were rewritten and validated:")
print(
    (
        ras_rewrite_failed_report["rewrite_status"]
        == "rewritten_and_validated"
    ).all()
)

## 1.12. Interpretation of Rewritten Non-RAS Outputs

The 5 originally non-RAS images that failed the stricter voxel-array validation were rewritten directly from their expected canonical RAS versions.

After rewriting, all 5 images passed validation. Their saved outputs are in RAS orientation, their shapes match the expected RAS shapes, their affine matrices match the expected RAS affine matrices, and their voxel arrays match exactly after reload.

This confirms that the non-RAS images have now been correctly standardized at both the header/affine level and the voxel-array level. The RAS-standardized output folder can now be treated as the orientation-standardized MRI dataset.

## 1.13. Final QC of RAS-Standardized NIfTI Dataset

After rewriting and validating the problematic non-RAS outputs, I perform a final QC check on the complete RAS-standardized dataset.

This step verifies that there is one RAS-standardized NIfTI file for every selected MRI scan, that all output files exist, that all files can be loaded, and that all output images report RAS orientation.

In [ ]:
FINAL_RAS_QC_REPORT_PATH = QC_DIR / "clean_90d_final_ras_nifti_qc_report.csv"
FINAL_RAS_MANIFEST_PATH = MANIFEST_DIR / "clean_90d_ras_nifti_manifest_1063.csv"

# Reload the original reorientation report if needed
if "ras_reorientation_report" not in globals():
    ras_reorientation_report = pd.read_csv(RAS_REORIENTATION_REPORT_PATH)

final_ras_records = []

for _, row in ras_reorientation_report.iterrows():
    output_path = Path(row["output_nifti_path"])

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "input_orientation": row["input_orientation"],
        "output_nifti_path": str(output_path),
        "output_file_exists": output_path.exists(),
        "load_status": None,
        "output_orientation": None,
        "output_shape": None,
        "voxel_sizes": None,
        "data_dtype": None,
        "file_size_mb": None,
        "final_ras_qc_status": None,
        "error_message": ""
    }

    try:
        img = nib.load(str(output_path))

        record["load_status"] = "loaded"
        record["output_orientation"] = "".join(nib.aff2axcodes(img.affine))
        record["output_shape"] = str(img.shape)
        record["voxel_sizes"] = str(img.header.get_zooms()[:len(img.shape)])
        record["data_dtype"] = str(img.get_data_dtype())
        record["file_size_mb"] = output_path.stat().st_size / (1024 ** 2)

        if (
            record["output_file_exists"]
            and record["load_status"] == "loaded"
            and record["output_orientation"] == "RAS"
            and len(img.shape) == 3
        ):
            record["final_ras_qc_status"] = "passed"
        else:
            record["final_ras_qc_status"] = "check_output"

    except Exception as e:
        record["load_status"] = "load_failed"
        record["final_ras_qc_status"] = "failed"
        record["error_message"] = str(e)

    final_ras_records.append(record)

final_ras_qc_report = pd.DataFrame(final_ras_records)
final_ras_qc_report.to_csv(FINAL_RAS_QC_REPORT_PATH, index=False)

# Create a clean manifest for the RAS-standardized NIfTI files
ras_manifest = extraction_manifest.merge(
    final_ras_qc_report[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "output_nifti_path",
            "output_orientation",
            "output_shape",
            "voxel_sizes",
            "data_dtype",
            "file_size_mb",
            "final_ras_qc_status"
        ]
    ],
    on=["RID", "subject_id", "final_group", "image_id"],
    how="left"
)

ras_manifest.to_csv(FINAL_RAS_MANIFEST_PATH, index=False)

print("Saved final RAS QC report:")
print(FINAL_RAS_QC_REPORT_PATH)

print("\nSaved final RAS NIfTI manifest:")
print(FINAL_RAS_MANIFEST_PATH)

print("\nFinal RAS QC status counts:")
print(final_ras_qc_report["final_ras_qc_status"].value_counts(dropna=False))

print("\nOutput file existence:")
print(final_ras_qc_report["output_file_exists"].value_counts(dropna=False))

print("\nLoad status:")
print(final_ras_qc_report["load_status"].value_counts(dropna=False))

print("\nOutput orientation counts:")
print(final_ras_qc_report["output_orientation"].value_counts(dropna=False))

print("\nOutput counts by final group:")
print(final_ras_qc_report["final_group"].value_counts())

print("\nUnique subjects:")
print(final_ras_qc_report["subject_id"].nunique())

print("\nUnique image IDs:")
print(final_ras_qc_report["image_id"].nunique())

print("\nFile size summary, MB:")
display(final_ras_qc_report["file_size_mb"].describe())

problem_final_ras = final_ras_qc_report[
    final_ras_qc_report["final_ras_qc_status"] != "passed"
].copy()

print("\nProblem final RAS outputs:")
print(len(problem_final_ras))

if len(problem_final_ras) > 0:
    display(problem_final_ras)

print("\nAll final RAS outputs passed QC:")
print((final_ras_qc_report["final_ras_qc_status"] == "passed").all())

## 1.14. Summarize Final RAS-Standardized Dataset

The RAS-standardized NIfTI dataset has passed orientation QC. Before applying further preprocessing, I summarize the remaining variation in image shape, voxel size, file size, scanner field strength, and clinical group distribution.

This step helps decide the next preprocessing target, such as the common voxel size and fixed image shape needed for machine learning.

In [ ]:
FINAL_RAS_MANIFEST_PATH = MANIFEST_DIR / "clean_90d_ras_nifti_manifest_1063.csv"

ras_manifest = pd.read_csv(FINAL_RAS_MANIFEST_PATH)

print("RAS manifest shape:")
print(ras_manifest.shape)

print("\nFinal group counts:")
print(ras_manifest["final_group"].value_counts())

print("\nFinal RAS QC status:")
print(ras_manifest["final_ras_qc_status"].value_counts(dropna=False))

print("\nOutput orientation counts:")
print(ras_manifest["output_orientation"].value_counts(dropna=False))

print("\nMost common output shapes:")
print(ras_manifest["output_shape"].value_counts().head(20))

print("\nMost common voxel sizes:")
print(ras_manifest["voxel_sizes"].value_counts().head(20))

print("\nData type counts:")
print(ras_manifest["data_dtype"].value_counts(dropna=False))

print("\nFile size summary, MB:")
display(ras_manifest["file_size_mb"].describe())

if "field_strength_category" in ras_manifest.columns:
    print("\nField strength counts:")
    print(ras_manifest["field_strength_category"].value_counts(dropna=False))

    print("\nField strength by final group:")
    display(
        ras_manifest
        .groupby(["final_group", "field_strength_category"])
        .size()
        .unstack(fill_value=0)
    )

print("\nUnique subjects:")
print(ras_manifest["subject_id"].nunique())

print("\nUnique image IDs:")
print(ras_manifest["image_id"].nunique())

print("\nPreview:")
display(ras_manifest.head())

## 1.15. Merge RAS Manifest with Clean Scanner Metadata

The RAS-standardized NIfTI manifest confirms that all 1063 images passed orientation QC. However, the current RAS manifest was created from the original extraction manifest, so it may not include the cleaned scanner metadata recovered from the NIfTI JSON sidecar files.

This step merges the final RAS NIfTI manifest with the JSON-derived scanner metadata. The resulting manifest keeps the RAS-standardized NIfTI paths and also includes complete scanner information such as cleaned field strength, manufacturer, scanner model, protocol name, and series description.

This merged manifest will be used as the source of truth for the next preprocessing stages.

In [ ]:
FINAL_RAS_MANIFEST_WITH_SCANNER_PATH = MANIFEST_DIR / "clean_90d_ras_nifti_manifest_1063_with_scanner_metadata.csv"

# Reload files if needed
ras_manifest = pd.read_csv(FINAL_RAS_MANIFEST_PATH)

if "scanner_metadata_clean" not in globals():
    CLEAN_SCANNER_METADATA_REPORT_PATH = QC_DIR / "clean_90d_standardized_scanner_metadata.csv"
    scanner_metadata_clean = pd.read_csv(CLEAN_SCANNER_METADATA_REPORT_PATH)

scanner_cols = [
    "RID",
    "subject_id",
    "final_group",
    "image_id",
    "field_strength_json_raw",
    "field_strength_t_clean",
    "field_strength_category",
    "manufacturer_json",
    "scanner_model_json",
    "protocol_name_json",
    "series_description_json"
]

ras_manifest_with_scanner = ras_manifest.merge(
    scanner_metadata_clean[scanner_cols],
    on=["RID", "subject_id", "final_group", "image_id"],
    how="left"
)

ras_manifest_with_scanner.to_csv(FINAL_RAS_MANIFEST_WITH_SCANNER_PATH, index=False)

print("Saved final RAS manifest with scanner metadata:")
print(FINAL_RAS_MANIFEST_WITH_SCANNER_PATH)

print("\nShape:")
print(ras_manifest_with_scanner.shape)

print("\nFinal group counts:")
print(ras_manifest_with_scanner["final_group"].value_counts())

print("\nRAS QC status:")
print(ras_manifest_with_scanner["final_ras_qc_status"].value_counts(dropna=False))

print("\nOrientation counts:")
print(ras_manifest_with_scanner["output_orientation"].value_counts(dropna=False))

print("\nClean field strength counts:")
print(ras_manifest_with_scanner["field_strength_category"].value_counts(dropna=False))

print("\nClean field strength by final group:")
display(
    ras_manifest_with_scanner
    .groupby(["final_group", "field_strength_category"])
    .size()
    .unstack(fill_value=0)
)

print("\nManufacturer counts:")
print(ras_manifest_with_scanner["manufacturer_json"].value_counts(dropna=False))

print("\nMissing scanner metadata checks:")
print("Missing field_strength_category:", ras_manifest_with_scanner["field_strength_category"].isna().sum())
print("Missing manufacturer_json:", ras_manifest_with_scanner["manufacturer_json"].isna().sum())
print("Missing scanner_model_json:", ras_manifest_with_scanner["scanner_model_json"].isna().sum())

print("\nPreview:")
display(
    ras_manifest_with_scanner[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "output_nifti_path",
            "output_shape",
            "voxel_sizes",
            "field_strength_category",
            "manufacturer_json",
            "scanner_model_json",
            "final_ras_qc_status"
        ]
    ].head()
)

## 1.16. Analyze RAS Image Geometry Before Resampling

The RAS-standardized images now have consistent orientation, but they still differ in image shape and voxel size. Before choosing a common preprocessing target, I need to inspect the physical geometry of the images.

This step loads each RAS-standardized NIfTI header and records:

- image shape
- voxel size in millimetres
- approximate physical field of view in millimetres

This helps decide whether to resample all images to a common voxel size, and what fixed image shape should be used later for deep learning.

In [ ]:
RAS_GEOMETRY_REPORT_PATH = QC_DIR / "clean_90d_ras_geometry_report.csv"

# Reload final RAS manifest with scanner metadata if needed
if "ras_manifest_with_scanner" not in globals():
    FINAL_RAS_MANIFEST_WITH_SCANNER_PATH = MANIFEST_DIR / "clean_90d_ras_nifti_manifest_1063_with_scanner_metadata.csv"
    ras_manifest_with_scanner = pd.read_csv(FINAL_RAS_MANIFEST_WITH_SCANNER_PATH)

geometry_records = []

for _, row in ras_manifest_with_scanner.iterrows():
    nifti_path = Path(row["output_nifti_path"])

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "nifti_path": str(nifti_path),
        "load_status": None,
        "shape_x": np.nan,
        "shape_y": np.nan,
        "shape_z": np.nan,
        "voxel_x_mm": np.nan,
        "voxel_y_mm": np.nan,
        "voxel_z_mm": np.nan,
        "fov_x_mm": np.nan,
        "fov_y_mm": np.nan,
        "fov_z_mm": np.nan,
        "orientation": None,
        "error_message": ""
    }

    try:
        img = nib.load(str(nifti_path))
        shape = img.shape
        zooms = img.header.get_zooms()[:3]

        record["load_status"] = "loaded"
        record["shape_x"] = shape[0]
        record["shape_y"] = shape[1]
        record["shape_z"] = shape[2]
        record["voxel_x_mm"] = float(zooms[0])
        record["voxel_y_mm"] = float(zooms[1])
        record["voxel_z_mm"] = float(zooms[2])
        record["fov_x_mm"] = shape[0] * float(zooms[0])
        record["fov_y_mm"] = shape[1] * float(zooms[1])
        record["fov_z_mm"] = shape[2] * float(zooms[2])
        record["orientation"] = "".join(nib.aff2axcodes(img.affine))

    except Exception as e:
        record["load_status"] = "load_failed"
        record["error_message"] = str(e)

    geometry_records.append(record)

ras_geometry_report = pd.DataFrame(geometry_records)
ras_geometry_report.to_csv(RAS_GEOMETRY_REPORT_PATH, index=False)

print("Saved RAS geometry report:")
print(RAS_GEOMETRY_REPORT_PATH)

print("\nLoad status counts:")
print(ras_geometry_report["load_status"].value_counts(dropna=False))

print("\nOrientation counts:")
print(ras_geometry_report["orientation"].value_counts(dropna=False))

print("\nShape summary:")
display(
    ras_geometry_report[
        ["shape_x", "shape_y", "shape_z"]
    ].describe()
)

print("\nVoxel size summary, mm:")
display(
    ras_geometry_report[
        ["voxel_x_mm", "voxel_y_mm", "voxel_z_mm"]
    ].describe()
)

print("\nPhysical field-of-view summary, mm:")
display(
    ras_geometry_report[
        ["fov_x_mm", "fov_y_mm", "fov_z_mm"]
    ].describe()
)

print("\nMost common voxel-size combinations:")
print(
    ras_geometry_report[
        ["voxel_x_mm", "voxel_y_mm", "voxel_z_mm"]
    ].value_counts().head(20)
)

print("\nMost common shape combinations:")
print(
    ras_geometry_report[
        ["shape_x", "shape_y", "shape_z"]
    ].value_counts().head(20)
)

print("\nGeometry by field strength:")
if "field_strength_category" in ras_manifest_with_scanner.columns:
    geometry_with_scanner = ras_geometry_report.merge(
        ras_manifest_with_scanner[
            ["RID", "subject_id", "final_group", "image_id", "field_strength_category"]
        ],
        on=["RID", "subject_id", "final_group", "image_id"],
        how="left"
    )

    display(
        geometry_with_scanner
        .groupby("field_strength_category")[
            ["shape_x", "shape_y", "shape_z", "voxel_x_mm", "voxel_y_mm", "voxel_z_mm"]
        ]
        .describe()
    )

print("\nProblem geometry rows:")
problem_geometry = ras_geometry_report[
    (ras_geometry_report["load_status"] != "loaded") |
    (ras_geometry_report["orientation"] != "RAS")
].copy()

print(len(problem_geometry))

if len(problem_geometry) > 0:
    display(problem_geometry)

## 1.17. Interpretation of RAS Image Geometry Report

The RAS geometry report was generated to inspect the remaining spatial variation in the orientation-standardized MRI dataset before resampling. All 1063 RAS-standardized NIfTI files loaded successfully, and all 1063 images were confirmed to be in RAS orientation. No problem geometry rows were identified, which confirms that the dataset is complete and geometrically valid at this stage.

Although orientation has been standardized, the images still vary in voxel dimensions and physical resolution. The image shapes range from `(124, 256, 256)` to `(211, 256, 256)`, with the most common shapes being `(176, 240, 256)`, `(160, 192, 192)`, `(208, 240, 256)`, and `(170, 256, 256)`. This confirms that the images cannot yet be used directly as uniform deep learning inputs, because deep learning models require consistent input dimensions.

The voxel-size summary also shows remaining scanner/protocol variation. The most common voxel-size combinations are close to 1.0 mm or 1.2 mm in one dimension with approximately 1.0-1.25 mm in the other dimensions. The voxel sizes are therefore broadly high-resolution structural MRI scans, but they are not identical across subjects.

The physical field-of-view values are generally comparable across the dataset, with median coverage of approximately 208 mm, 240 mm, and 256 mm across the three axes. However, there is still enough variation to require careful resampling and later crop/pad standardization.

The geometry summary by scanner field strength shows that 1.5T and 3T scans have different typical image geometry. The 1.5T scans tend to have smaller image dimensions and coarser voxel spacing, while the 3T scans tend to have larger image dimensions and finer voxel spacing. This confirms that scanner/protocol differences remain present in the image data and should be considered during preprocessing and later analysis.

This output confirms that the RAS-standardized dataset is valid and ready for the next preprocessing stage. The next step is to estimate candidate resampling targets, such as 1.0 mm, 1.5 mm, and 2.0 mm isotropic voxel spacing, before choosing a practical target for deep learning.


## 1.18. Estimate Resampling Targets Before Processing

The RAS-standardized images have consistent orientation, but they still differ in voxel size and physical field of view. Before actually resampling the images, I estimate the output image dimensions that would result from several candidate voxel sizes.

This step compares 1.0 mm, 1.5 mm, and 2.0 mm isotropic resampling targets. The goal is to choose a target that preserves enough anatomical information while keeping the image size practical for deep learning.

In [ ]:
RESAMPLING_TARGET_ESTIMATE_PATH = QC_DIR / "clean_90d_resampling_target_size_estimates.csv"

if "ras_geometry_report" not in globals():
    RAS_GEOMETRY_REPORT_PATH = QC_DIR / "clean_90d_ras_geometry_report.csv"
    ras_geometry_report = pd.read_csv(RAS_GEOMETRY_REPORT_PATH)

candidate_voxel_sizes = [1.0, 1.5, 2.0]

estimate_records = []

for target_voxel_size in candidate_voxel_sizes:
    temp = ras_geometry_report.copy()

    temp["target_voxel_size_mm"] = target_voxel_size

    temp["estimated_shape_x"] = np.ceil(temp["fov_x_mm"] / target_voxel_size).astype(int)
    temp["estimated_shape_y"] = np.ceil(temp["fov_y_mm"] / target_voxel_size).astype(int)
    temp["estimated_shape_z"] = np.ceil(temp["fov_z_mm"] / target_voxel_size).astype(int)

    temp["estimated_total_voxels"] = (
        temp["estimated_shape_x"]
        * temp["estimated_shape_y"]
        * temp["estimated_shape_z"]
    )

    temp["estimated_float32_mb_per_image"] = (
        temp["estimated_total_voxels"] * 4 / (1024 ** 2)
    )

    estimate_records.append(temp)

resampling_target_estimates = pd.concat(estimate_records, ignore_index=True)
resampling_target_estimates.to_csv(RESAMPLING_TARGET_ESTIMATE_PATH, index=False)

print("Saved resampling target estimate report:")
print(RESAMPLING_TARGET_ESTIMATE_PATH)

for target_voxel_size in candidate_voxel_sizes:
    subset = resampling_target_estimates[
        resampling_target_estimates["target_voxel_size_mm"] == target_voxel_size
    ].copy()

    print("\n" + "=" * 80)
    print(f"Target voxel size: {target_voxel_size} mm isotropic")
    print("=" * 80)

    print("\nEstimated output shape summary:")
    display(
        subset[
            ["estimated_shape_x", "estimated_shape_y", "estimated_shape_z"]
        ].describe()
    )

    print("\nEstimated float32 memory per image, MB:")
    display(subset["estimated_float32_mb_per_image"].describe())

    print("\nEstimated full dataset size as float32 arrays, GB:")
    total_gb = subset["estimated_float32_mb_per_image"].sum() / 1024
    print(round(total_gb, 2))

print("\nLargest estimated images for 2.0 mm target:")
display(
    resampling_target_estimates[
        resampling_target_estimates["target_voxel_size_mm"] == 2.0
    ]
    .sort_values("estimated_total_voxels", ascending=False)
    [
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "fov_x_mm",
            "fov_y_mm",
            "fov_z_mm",
            "estimated_shape_x",
            "estimated_shape_y",
            "estimated_shape_z",
            "estimated_float32_mb_per_image"
        ]
    ]
    .head(10)
)

## 1.19. Interpretation of Resampling Size Estimates

The resampling target estimate compared three candidate isotropic voxel sizes: 1.0 mm, 1.5 mm, and 2.0 mm. This step did not modify the images. It only estimated the output image dimensions and approximate memory requirements that would result from each candidate voxel size.

The 1.0 mm isotropic target preserves the highest spatial detail but produces the largest arrays. At this resolution, the estimated median image shape is approximately `(208, 240, 256)`, with an average memory requirement of about 49 MB per image when stored as uncompressed `float32`. Across all 1063 images, this would require approximately 51 GB as float32 arrays. This is much larger than the compressed image files currently stored on disk because deep learning arrays are typically loaded as uncompressed floating-point values.

The 1.5 mm isotropic target provides a middle-ground option. It reduces the estimated median image size to approximately `(139, 160, 171)`, with an average memory requirement of about 14.6 MB per image. The estimated full dataset size is approximately 15.16 GB as float32 arrays. This option preserves more spatial detail than 2.0 mm, but it is still substantially heavier than 2.0 mm for model training.

The 2.0 mm isotropic target is the most computationally practical option among the three candidates. It reduces the estimated median image size to approximately `(104, 120, 128)`, with an average memory requirement of about 6.16 MB per image. The estimated full dataset size is approximately 6.4 GB as float32 arrays. This is much more manageable for a first deep learning baseline.

The difference between 1.0 mm and 2.0 mm is large because MRI data are three-dimensional. Halving the voxel size from 2.0 mm to 1.0 mm approximately doubles the number of voxels along each axis, which increases the total number of voxels by about eight times. This explains why the estimated full dataset size increases from about 6.4 GB at 2.0 mm to about 51 GB at 1.0 mm.


## 1.20. Choice of Resampling Target

Based on the resampling size estimates, I selected 1.5 mm isotropic voxel spacing as the target resolution for the next preprocessing stage.

The 1.0 mm isotropic target preserves the highest spatial detail, but it would produce large uncompressed float32 arrays, with an estimated full-dataset size of approximately 51 GB. This would be computationally expensive for a first deep learning pipeline, especially for 3D image-based modelling.

The 2.0 mm isotropic target is the most memory-efficient option, with an estimated full-dataset size of approximately 6.4 GB. However, it also reduces spatial detail more aggressively, which may remove subtle anatomical information relevant to Alzheimer’s disease and MCI conversion.

The 1.5 mm isotropic target provides a practical compromise. It reduces the expected dataset size to approximately 15.16 GB as float32 arrays while preserving more anatomical detail than 2.0 mm. This makes it suitable for building a feasible first 3D MRI preprocessing pipeline while remaining closer to the spatial detail used in stronger neuroimaging studies.


## 1.21. Pilot Resampling

In [ ]:
from nibabel.processing import resample_to_output

RESAMPLED_1P5MM_DIR = PROCESSED_DIR / "nifti_1p5mm"
PILOT_RESAMPLING_REPORT_PATH = QC_DIR / "clean_90d_pilot_resampling_1p5mm_report.csv"

RESAMPLED_1P5MM_DIR.mkdir(parents=True, exist_ok=True)

TARGET_VOXEL_SIZE_MM = 1.5

# Reload final RAS manifest with scanner metadata if needed
if "ras_manifest_with_scanner" not in globals():
    FINAL_RAS_MANIFEST_WITH_SCANNER_PATH = MANIFEST_DIR / "clean_90d_ras_nifti_manifest_1063_with_scanner_metadata.csv"
    ras_manifest_with_scanner = pd.read_csv(FINAL_RAS_MANIFEST_WITH_SCANNER_PATH)

# Select a small pilot set covering different original shapes/groups
pilot_cases = (
    ras_manifest_with_scanner
    .sort_values(["final_group", "output_shape"])
    .groupby("final_group")
    .head(1)
    .copy()
)

print("Pilot cases selected:", len(pilot_cases))
display(
    pilot_cases[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "output_shape",
            "voxel_sizes",
            "output_nifti_path"
        ]
    ]
)

pilot_resampling_records = []

for _, row in pilot_cases.iterrows():
    input_path = Path(row["output_nifti_path"])
    group_dir = RESAMPLED_1P5MM_DIR / row["final_group"]
    group_dir.mkdir(parents=True, exist_ok=True)

    output_path = group_dir / f"{row['subject_id']}_I{int(row['image_id'])}_1p5mm.nii.gz"

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "input_path": str(input_path),
        "output_path": str(output_path),
        "input_shape": None,
        "input_voxel_sizes": None,
        "output_shape": None,
        "output_voxel_sizes": None,
        "output_orientation": None,
        "output_dtype": None,
        "output_file_size_mb": None,
        "resampling_status": None,
        "error_message": ""
    }

    try:
        input_img = nib.load(str(input_path))

        record["input_shape"] = str(input_img.shape)
        record["input_voxel_sizes"] = str(input_img.header.get_zooms()[:3])

        # order=1 means linear interpolation, appropriate for continuous MRI intensity images.
        resampled_img = resample_to_output(
            input_img,
            voxel_sizes=(TARGET_VOXEL_SIZE_MM, TARGET_VOXEL_SIZE_MM, TARGET_VOXEL_SIZE_MM),
            order=1
        )

        # Save as float32 to make the later deep learning pipeline consistent.
        resampled_data = resampled_img.get_fdata(dtype=np.float32)
        resampled_img_float32 = nib.Nifti1Image(
            resampled_data,
            resampled_img.affine
        )
        resampled_img_float32.header.set_data_dtype(np.float32)
        resampled_img_float32.set_qform(resampled_img.affine, code=1)
        resampled_img_float32.set_sform(resampled_img.affine, code=1)

        nib.save(resampled_img_float32, str(output_path))

        saved_img = nib.load(str(output_path))

        record["output_shape"] = str(saved_img.shape)
        record["output_voxel_sizes"] = str(saved_img.header.get_zooms()[:3])
        record["output_orientation"] = "".join(nib.aff2axcodes(saved_img.affine))
        record["output_dtype"] = str(saved_img.get_data_dtype())
        record["output_file_size_mb"] = output_path.stat().st_size / (1024 ** 2)

        if (
            record["output_orientation"] == "RAS"
            and record["output_dtype"] == "float32"
            and all(abs(v - TARGET_VOXEL_SIZE_MM) < 0.01 for v in saved_img.header.get_zooms()[:3])
        ):
            record["resampling_status"] = "passed"
        else:
            record["resampling_status"] = "check_output"

    except Exception as e:
        record["resampling_status"] = "failed"
        record["error_message"] = str(e)

    pilot_resampling_records.append(record)

pilot_resampling_report = pd.DataFrame(pilot_resampling_records)
pilot_resampling_report.to_csv(PILOT_RESAMPLING_REPORT_PATH, index=False)

print("Saved pilot resampling report:")
print(PILOT_RESAMPLING_REPORT_PATH)

print("\nPilot resampling status counts:")
print(pilot_resampling_report["resampling_status"].value_counts(dropna=False))

print("\nOutput orientation counts:")
print(pilot_resampling_report["output_orientation"].value_counts(dropna=False))

print("\nOutput dtype counts:")
print(pilot_resampling_report["output_dtype"].value_counts(dropna=False))

print("\nOutput file size summary, MB:")
display(pilot_resampling_report["output_file_size_mb"].describe())

print("\nPilot resampling details:")
display(pilot_resampling_report)

print("\nAll pilot resampling outputs passed QC:")
print((pilot_resampling_report["resampling_status"] == "passed").all())

## 1.22. Interpretation of Pilot 1.5 mm Resampling

A pilot resampling step was performed before applying 1.5 mm isotropic resampling to the full MRI dataset. Four images were selected, one from each final diagnostic group: AD, CN, pMCI, and sMCI.

All four pilot images were successfully resampled. The outputs passed QC, remained in RAS orientation, and were saved as `float32` NIfTI files. The output voxel sizes were confirmed to be 1.5 mm isotropic in all three spatial dimensions.

The output shapes decreased compared with the original RAS-standardized images, which is expected because several original images had voxel sizes smaller than 1.5 mm. Resampling to 1.5 mm increases voxel spacing and therefore reduces the number of voxels needed to represent approximately the same physical field of view.

The saved `.nii.gz` file sizes were smaller than the earlier memory estimates because NIfTI files are compressed on disk. The earlier estimates represented approximate uncompressed `float32` array sizes for deep learning, while the pilot output file sizes represent compressed storage size.

Overall, the pilot confirms that 1.5 mm isotropic resampling is technically working and produces valid RAS-oriented outputs. Before running the full dataset, the next step is to visually inspect the pilot resampled images to confirm that the anatomical structure remains reasonable after interpolation.


## 1.23. Visual Check

In [ ]:
import matplotlib.pyplot as plt

if "pilot_resampling_report" not in globals():
    PILOT_RESAMPLING_REPORT_PATH = QC_DIR / "clean_90d_pilot_resampling_1p5mm_report.csv"
    pilot_resampling_report = pd.read_csv(PILOT_RESAMPLING_REPORT_PATH)

def normalize_for_display(slice_2d):
    lower, upper = np.percentile(slice_2d, [1, 99])
    if upper <= lower:
        return slice_2d
    normalized = (slice_2d - lower) / (upper - lower)
    return np.clip(normalized, 0, 1)

def get_middle_slices(data):
    x_mid = data.shape[0] // 2
    y_mid = data.shape[1] // 2
    z_mid = data.shape[2] // 2

    sagittal = data[x_mid, :, :]
    coronal = data[:, y_mid, :]
    axial = data[:, :, z_mid]

    return sagittal, coronal, axial

for _, row in pilot_resampling_report.iterrows():
    original_path = Path(row["input_path"])
    resampled_path = Path(row["output_path"])

    original_img = nib.load(str(original_path))
    resampled_img = nib.load(str(resampled_path))

    original_data = original_img.get_fdata(dtype=np.float32)
    resampled_data = resampled_img.get_fdata(dtype=np.float32)

    original_sagittal, original_coronal, original_axial = get_middle_slices(original_data)
    resampled_sagittal, resampled_coronal, resampled_axial = get_middle_slices(resampled_data)

    original_slices = [
        normalize_for_display(original_axial),
        normalize_for_display(original_coronal),
        normalize_for_display(original_sagittal)
    ]

    resampled_slices = [
        normalize_for_display(resampled_axial),
        normalize_for_display(resampled_coronal),
        normalize_for_display(resampled_sagittal)
    ]

    view_names = ["Axial", "Coronal", "Sagittal"]

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))

    for i, view_name in enumerate(view_names):
        axes[0, i].imshow(np.rot90(original_slices[i]), cmap="gray")
        axes[0, i].set_title(f"Original RAS\n{view_name}")
        axes[0, i].axis("off")

        axes[1, i].imshow(np.rot90(resampled_slices[i]), cmap="gray")
        axes[1, i].set_title(f"Resampled 1.5 mm\n{view_name}")
        axes[1, i].axis("off")

    fig.suptitle(
        f"{row['subject_id']} | I{int(row['image_id'])} | {row['final_group']} | "
        f"{row['input_shape']} → {row['output_shape']}",
        fontsize=11
    )

    plt.tight_layout()
    plt.show()

## 1.24. Visual QC of Pilot 1.5 mm Resampling

The pilot 1.5 mm resampling outputs were visually inspected before applying the same preprocessing step to the full MRI dataset.

The original RAS image and the resampled 1.5 mm image were compared in axial, coronal, and sagittal views. The resampled image appears slightly smoother, which is expected because linear interpolation is used during resampling. However, the overall anatomical structure is preserved. There is no visible evidence of left-right flipping, rotation error, severe blurring, missing brain tissue, or corrupted/empty output.

Together with the quantitative pilot QC, this visual inspection supports proceeding with 1.5 mm isotropic resampling for the full RAS-standardized MRI dataset.


In [ ]:
from nibabel.processing import resample_to_output
import time

FULL_RESAMPLING_1P5MM_REPORT_PATH = QC_DIR / "clean_90d_full_resampling_1p5mm_report.csv"
RESAMPLED_1P5MM_MANIFEST_PATH = MANIFEST_DIR / "clean_90d_resampled_1p5mm_manifest_1063.csv"

RESAMPLED_1P5MM_DIR = PROCESSED_DIR / "nifti_1p5mm"
RESAMPLED_1P5MM_DIR.mkdir(parents=True, exist_ok=True)

TARGET_VOXEL_SIZE_MM = 1.5
OVERWRITE_EXISTING = False

# Reload final RAS manifest with scanner metadata if needed
if "ras_manifest_with_scanner" not in globals():
    FINAL_RAS_MANIFEST_WITH_SCANNER_PATH = MANIFEST_DIR / "clean_90d_ras_nifti_manifest_1063_with_scanner_metadata.csv"
    ras_manifest_with_scanner = pd.read_csv(FINAL_RAS_MANIFEST_WITH_SCANNER_PATH)

print("Images to process:", len(ras_manifest_with_scanner))
print("Target voxel size:", TARGET_VOXEL_SIZE_MM, "mm isotropic")
print("Output folder:", RESAMPLED_1P5MM_DIR)
print("Overwrite existing outputs:", OVERWRITE_EXISTING)

full_resampling_records = []
start_time = time.time()

def validate_resampled_output(output_path, target_voxel_size):
    """
    Load an existing or newly saved resampled image and check basic QC.
    """
    img = nib.load(str(output_path))
    zooms = img.header.get_zooms()[:3]
    orientation = "".join(nib.aff2axcodes(img.affine))
    dtype = str(img.get_data_dtype())

    voxel_size_ok = all(abs(float(v) - target_voxel_size) < 0.01 for v in zooms)

    passed = (
        orientation == "RAS"
        and dtype == "float32"
        and voxel_size_ok
        and len(img.shape) == 3
    )

    return img, zooms, orientation, dtype, passed

for i, (_, row) in enumerate(ras_manifest_with_scanner.iterrows(), start=1):
    input_path = Path(row["output_nifti_path"])

    group_dir = RESAMPLED_1P5MM_DIR / row["final_group"]
    group_dir.mkdir(parents=True, exist_ok=True)

    output_path = group_dir / f"{row['subject_id']}_I{int(row['image_id'])}_1p5mm.nii.gz"

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "input_path": str(input_path),
        "output_path": str(output_path),
        "input_shape": None,
        "input_voxel_sizes": None,
        "output_shape": None,
        "output_voxel_sizes": None,
        "output_orientation": None,
        "output_dtype": None,
        "output_file_size_mb": None,
        "resampling_status": None,
        "error_message": ""
    }

    try:
        input_img = nib.load(str(input_path))
        record["input_shape"] = str(input_img.shape)
        record["input_voxel_sizes"] = str(input_img.header.get_zooms()[:3])

        # If a valid output already exists, keep it.
        # This avoids unnecessarily rewriting the pilot images.
        if output_path.exists() and not OVERWRITE_EXISTING:
            saved_img, zooms, orientation, dtype, passed = validate_resampled_output(
                output_path,
                TARGET_VOXEL_SIZE_MM
            )

            record["output_shape"] = str(saved_img.shape)
            record["output_voxel_sizes"] = str(zooms)
            record["output_orientation"] = orientation
            record["output_dtype"] = dtype
            record["output_file_size_mb"] = output_path.stat().st_size / (1024 ** 2)

            if passed:
                record["resampling_status"] = "passed_existing"
            else:
                record["resampling_status"] = "existing_failed_qc_rewritten"

        # Resample if output does not exist, overwrite is requested, or existing file failed QC.
        if (
            not output_path.exists()
            or OVERWRITE_EXISTING
            or record["resampling_status"] == "existing_failed_qc_rewritten"
        ):
            # order=1 means linear interpolation, appropriate for continuous MRI intensity images.
            resampled_img = resample_to_output(
                input_img,
                voxel_sizes=(
                    TARGET_VOXEL_SIZE_MM,
                    TARGET_VOXEL_SIZE_MM,
                    TARGET_VOXEL_SIZE_MM
                ),
                order=1
            )

            # Save as float32 for consistency in later deep learning preprocessing.
            resampled_data = resampled_img.get_fdata(dtype=np.float32)
            resampled_img_float32 = nib.Nifti1Image(
                resampled_data,
                resampled_img.affine
            )
            resampled_img_float32.header.set_data_dtype(np.float32)
            resampled_img_float32.set_qform(resampled_img.affine, code=1)
            resampled_img_float32.set_sform(resampled_img.affine, code=1)

            nib.save(resampled_img_float32, str(output_path))

            saved_img, zooms, orientation, dtype, passed = validate_resampled_output(
                output_path,
                TARGET_VOXEL_SIZE_MM
            )

            record["output_shape"] = str(saved_img.shape)
            record["output_voxel_sizes"] = str(zooms)
            record["output_orientation"] = orientation
            record["output_dtype"] = dtype
            record["output_file_size_mb"] = output_path.stat().st_size / (1024 ** 2)

            if passed:
                if record["resampling_status"] == "existing_failed_qc_rewritten":
                    record["resampling_status"] = "rewritten_and_passed"
                else:
                    record["resampling_status"] = "resampled_and_passed"
            else:
                record["resampling_status"] = "check_output"

    except Exception as e:
        record["resampling_status"] = "failed"
        record["error_message"] = str(e)

    full_resampling_records.append(record)

    if i % 25 == 0 or i == len(ras_manifest_with_scanner):
        elapsed_min = (time.time() - start_time) / 60
        print(f"Processed {i}/{len(ras_manifest_with_scanner)} images | elapsed: {elapsed_min:.1f} min")

full_resampling_report = pd.DataFrame(full_resampling_records)
full_resampling_report.to_csv(FULL_RESAMPLING_1P5MM_REPORT_PATH, index=False)

# Merge the resampled paths and QC fields into a model-preprocessing manifest
resampled_1p5mm_manifest = ras_manifest_with_scanner.merge(
    full_resampling_report[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "output_path",
            "output_shape",
            "output_voxel_sizes",
            "output_orientation",
            "output_dtype",
            "output_file_size_mb",
            "resampling_status"
        ]
    ],
    on=["RID", "subject_id", "final_group", "image_id"],
    how="left",
    suffixes=("", "_resampled")
)

resampled_1p5mm_manifest = resampled_1p5mm_manifest.rename(
    columns={
        "output_path": "resampled_1p5mm_nifti_path",
        "output_shape_resampled": "resampled_1p5mm_shape",
        "output_voxel_sizes": "resampled_1p5mm_voxel_sizes",
        "output_orientation_resampled": "resampled_1p5mm_orientation",
        "output_dtype": "resampled_1p5mm_dtype",
        "output_file_size_mb": "resampled_1p5mm_file_size_mb"
    }
)

resampled_1p5mm_manifest.to_csv(RESAMPLED_1P5MM_MANIFEST_PATH, index=False)

print("\nSaved full 1.5 mm resampling report:")
print(FULL_RESAMPLING_1P5MM_REPORT_PATH)

print("\nSaved 1.5 mm resampled manifest:")
print(RESAMPLED_1P5MM_MANIFEST_PATH)

print("\nResampling status counts:")
print(full_resampling_report["resampling_status"].value_counts(dropna=False))

print("\nOutput orientation counts:")
print(full_resampling_report["output_orientation"].value_counts(dropna=False))

print("\nOutput dtype counts:")
print(full_resampling_report["output_dtype"].value_counts(dropna=False))

print("\nOutput shape counts, top 20:")
print(full_resampling_report["output_shape"].value_counts().head(20))

print("\nOutput file size summary, MB:")
display(full_resampling_report["output_file_size_mb"].describe())

problem_resampling = full_resampling_report[
    ~full_resampling_report["resampling_status"].isin(
        ["resampled_and_passed", "passed_existing", "rewritten_and_passed"]
    )
].copy()

print("\nProblem resampling outputs:")
print(len(problem_resampling))

if len(problem_resampling) > 0:
    display(problem_resampling)

print("\nAll full 1.5 mm resampling outputs passed QC:")
print(len(problem_resampling) == 0)